# Phase 2.3: PubMed Abstract Acquisition

This notebook reads Phase 2 search results, skips PMIDs already in the raw article cache, fetches a bounded batch when enabled, and rebuilds the Phase 2 tables from cached records. Each successful MEDLINE record is appended immediately, so an interrupted run can resume.


In [5]:
from pathlib import Path
import sys

from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v2").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathogen_atlas_v2.abstracts.abstract_exports import (
    _read_jsonl_resume_records,
    _output_paths,
    build_pubmed_client,
    prepare_phase2_search_results,
    run_pubmed_fetches,
    selected_pmids_from_search_results,
    write_phase2_outputs,
)
from pathogen_atlas_v2.abstracts.pubmed_client import load_ncbi_entrez_config
from pathogen_atlas_v2.abstracts.pubmed_query_plan import load_pubmed_query_plan
from pathogen_atlas_v2.abstracts.pubmed_query_templates import load_pubmed_query_template_config
from pathogen_atlas_v2.common.io import read_jsonl, resolve_path
from pathogen_atlas_v2 import notebook_utils as nbutils


In [6]:
NOTEBOOK_CONFIG_PATH = "configs/notebook_parameters.yaml"
PARAMS = nbutils.notebook_phase_parameters("phase2_abstract_fetch", NOTEBOOK_CONFIG_PATH)

TEMPLATE_CONFIG_PATH = PARAMS["template_config_path"]
FILTER_CONFIG_PATH = PARAMS["filter_config_path"]
FORCE_REFRESH = bool(PARAMS["force_refresh"])
RUN_ABSTRACT_FETCH = bool(PARAMS["run_abstract_fetch"])
LOG_EVERY = int(PARAMS["log_every"])

PHASE2_CONFIG = load_pubmed_query_template_config(TEMPLATE_CONFIG_PATH)
PHASE2_PATHS = _output_paths(PHASE2_CONFIG)
QUERY_PLAN_PATH = resolve_path(PHASE2_CONFIG["query_plan"]["path"])
SEARCH_RESULTS_PATH = PHASE2_PATHS["raw_search_results_path"]
ARTICLE_CACHE_PATH = PHASE2_PATHS["raw_article_records_path"]
MAX_PMIDS_THIS_RUN = PHASE2_CONFIG["source"].get("max_pmids_per_fetch_run", 25)
FETCH_BATCH_SIZE = int(PHASE2_CONFIG["source"].get("fetch_batch_size", 200))


In [7]:
# Preflight: count selected IDs, cached records, remaining work, and request estimates.
if not SEARCH_RESULTS_PATH.exists():
    raise FileNotFoundError(f"Missing {SEARCH_RESULTS_PATH}. Run notebook 02.2 first.")
if not QUERY_PLAN_PATH.exists():
    raise FileNotFoundError(f"Missing {QUERY_PLAN_PATH}. Run notebook 02.1 first.")

query_plan = load_pubmed_query_plan(QUERY_PLAN_PATH)
search_results = _read_jsonl_resume_records(SEARCH_RESULTS_PATH)
search_results, _, _ = prepare_phase2_search_results(query_plan, search_results, TEMPLATE_CONFIG_PATH)
selected_pmids = selected_pmids_from_search_results(search_results)
article_records = _read_jsonl_resume_records(ARTICLE_CACHE_PATH) if ARTICLE_CACHE_PATH.exists() and not FORCE_REFRESH else []
cached_pmids = {str(record.get("PMID")) for record in article_records if record.get("PMID")}
pending_pmids = [pmid for pmid in selected_pmids if pmid not in cached_pmids]
scheduled_count = len(pending_pmids) if MAX_PMIDS_THIS_RUN is None else min(len(pending_pmids), int(MAX_PMIDS_THIS_RUN))
request_count = (scheduled_count + FETCH_BATCH_SIZE - 1) // FETCH_BATCH_SIZE
ncbi_config = load_ncbi_entrez_config(PHASE2_CONFIG["source"]["ncbi_config_path"])
rate_key = "rate_limit_with_api_key_per_second" if ncbi_config.api_key else "rate_limit_without_api_key_per_second"
configured_rate = float(PHASE2_CONFIG["source"][rate_key])
request_delay = PHASE2_CONFIG["source"].get("request_delay_seconds")
effective_rate = min(configured_rate, (1 / float(request_delay)) if request_delay else configured_rate)

fetch_preflight = {
    "selected_pmids": len(selected_pmids),
    "cached_article_records": len(cached_pmids),
    "pending_pmids": len(pending_pmids),
    "pmids_in_this_run_if_enabled": scheduled_count,
    "estimated_efetch_requests": request_count,
    "ncbi_api_key_configured": bool(ncbi_config.api_key),
    "effective_rate_limit_per_second": effective_rate,
    "estimated_minimum_request_time_seconds": round(request_count / effective_rate, 1),
    "run_abstract_fetch": RUN_ABSTRACT_FETCH,
    "force_refresh": FORCE_REFRESH,
}
display(fetch_preflight)


{'selected_pmids': 110216,
 'cached_article_records': 17225,
 'pending_pmids': 92991,
 'pmids_in_this_run_if_enabled': 92991,
 'estimated_efetch_requests': 465,
 'ncbi_api_key_configured': True,
 'effective_rate_limit_per_second': 10.0,
 'estimated_minimum_request_time_seconds': 46.5,
 'run_abstract_fetch': True,
 'force_refresh': False}

In [ ]:
if RUN_ABSTRACT_FETCH:
    if FORCE_REFRESH and ARTICLE_CACHE_PATH.exists():
        ARTICLE_CACHE_PATH.unlink()
        article_records = []

    # The client uses the email and optional NCBI API key, with a rate-aware delay.
    pubmed_client = build_pubmed_client(PHASE2_CONFIG)
    max_pmids = int(MAX_PMIDS_THIS_RUN) if MAX_PMIDS_THIS_RUN is not None else None
    article_records = run_pubmed_fetches(
        search_results=search_results,
        client=pubmed_client,
        existing_records=article_records,
        output_path=ARTICLE_CACHE_PATH,
        batch_size=FETCH_BATCH_SIZE,
        max_pmids=max_pmids,
        log=True,
        log_every=LOG_EVERY,
    )
    # All fetched records, including abstracts later judged unhelpful, remain cached.
    abstract_result = write_phase2_outputs(
        query_plan,
        search_results,
        article_records,
        TEMPLATE_CONFIG_PATH,
        FILTER_CONFIG_PATH,
        raw_article_records_already_appended=True,
        raw_search_results_already_saved=True,
    )
else:
    abstract_result = {"status": "not_run", "reason": "Set run_abstract_fetch: true in configs/notebook_parameters.yaml to fetch the next batch."}

abstract_summary = nbutils.phase_run_summary(abstract_result)
display(abstract_summary["overview"])
display(abstract_summary["counts"])
display(nbutils.metadata_outputs(abstract_result))


Abstract fetch: 17225/110216 PMIDs already fetched; 92991 remaining; 92991 scheduled this run.
Abstract fetch: 17425/110216 PMIDs fetched; 92791 remaining; 200/92991 attempted this run. ETA 28m 19s (about 2026-10-29 23:48:14).


In [ ]:
display(nbutils.read_table_if_exists(PHASE2_PATHS["release_abstracts_path"], n_rows=10))


In [ ]:
display(nbutils.read_table_if_exists(PHASE2_PATHS["release_species_matches_path"], n_rows=10))
